1. Downloading dataset

In [35]:
from pathlib import Path
import kagglehub
downloaded_dataset_directory = kagglehub.dataset_download(
    "annavictoria/speed-dating-experiment",
    output_dir="data",
)
speed_dating_csv_path = next(Path(downloaded_dataset_directory).rglob("*.csv"))

print("Downloaded data path:", speed_dating_csv_path)


Downloaded data path: data/Speed Dating Data.csv


2. Loading data

In [36]:
import pandas as pd

speed_dating_data = pd.read_csv(speed_dating_csv_path, encoding="latin-1")
speed_dating_data.shape


(8378, 195)

3. Dataset and target

Speed-dating events, Columbia University, 2002–2004. One row is one participant's decision after one meeting.

Target: `dec`. `1` = wants another meeting, `0` = does not.


4. Size

In [37]:
participant_partner_pairs = speed_dating_data.loc[
    speed_dating_data["pid"].notna(), ["iid", "pid"]
].drop_duplicates()

print("rows", len(speed_dating_data))
print("columns", speed_dating_data.shape[1])
print("participants", speed_dating_data["iid"].nunique())
print("meetings", len(participant_partner_pairs) // 2)
print("waves", speed_dating_data["wave"].nunique())


rows 8378
columns 195
participants 551
meetings 4184
waves 21


5. Feature count

In [38]:
identifier_columns = ["iid", "id", "idg", "partner", "pid", "wave"]
target_column = "dec"
scorecard_columns_written_together_with_the_decision = [
    "match", "dec_o", "match_es",
    "attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob", "met",
    "attr_o", "sinc_o", "intel_o", "fun_o", "amb_o", "shar_o", "like_o", "prob_o", "met_o",
]
followup_survey_columns = ["length", "you_call", "them_cal"]

feature_columns = [
    column
    for column in speed_dating_data.columns
    if column not in identifier_columns
    + [target_column]
    + scorecard_columns_written_together_with_the_decision
    + followup_survey_columns
    and not column.endswith(("_s", "_2", "_3"))
]

later_survey_column_count = sum(
    column.endswith(("_s", "_2", "_3")) or column in followup_survey_columns
    for column in speed_dating_data.columns
)

print("all columns", speed_dating_data.shape[1])
print("removed ids", len(identifier_columns))
print("removed target", 1)
print("removed scorecard columns", len(scorecard_columns_written_together_with_the_decision))
print("removed later surveys", later_survey_column_count)
print("features", len(feature_columns))
print("numeric features", speed_dating_data[feature_columns].select_dtypes(include="number").shape[1])
print("text features", speed_dating_data[feature_columns].select_dtypes(exclude="number").shape[1])


all columns 195
removed ids 6
removed target 1
removed scorecard columns 21
removed later surveys 87
features 80
numeric features 72
text features 8


6. Feature list

In [39]:
for column in feature_columns:
    print(column)


gender
condtn
round
position
positin1
order
int_corr
samerace
age_o
race_o
pf_o_att
pf_o_sin
pf_o_int
pf_o_fun
pf_o_amb
pf_o_sha
age
field
field_cd
undergra
mn_sat
tuition
race
imprace
imprelig
from
zipcode
income
goal
date
go_out
career
career_c
sports
tvsports
exercise
dining
museums
art
hiking
gaming
clubbing
reading
tv
theater
movies
concerts
music
shopping
yoga
exphappy
expnum
attr1_1
sinc1_1
intel1_1
fun1_1
amb1_1
shar1_1
attr4_1
sinc4_1
intel4_1
fun4_1
amb4_1
shar4_1
attr2_1
sinc2_1
intel2_1
fun2_1
amb2_1
shar2_1
attr3_1
sinc3_1
fun3_1
intel3_1
amb3_1
attr5_1
sinc5_1
intel5_1
fun5_1
amb5_1


7. Missing values

In [40]:
feature_values = speed_dating_data[feature_columns]
feature_cell_count = feature_values.size
empty_feature_cell_count = int(feature_values.isna().sum().sum())

print("feature cells", feature_cell_count)
print("empty cells", empty_feature_cell_count)
print("empty share", empty_feature_cell_count / feature_cell_count)
print("rows with a gap", int(feature_values.isna().any(axis="columns").sum()))

empty_share_by_feature = feature_values.isna().mean().sort_values(ascending=False)
display(empty_share_by_feature[empty_share_by_feature > 0].rename("empty_share"))


feature cells 670240
empty cells 60774
empty share 0.09067498209596563
rows with a gap 8378


expnum      0.785152
mn_sat      0.626044
tuition     0.572332
income      0.489258
amb5_1      0.414419
              ...   
fun2_1      0.009429
intel1_1    0.009429
race_o      0.008713
race        0.007520
field       0.007520
Name: empty_share, Length: 74, dtype: float64

8. Erroneous values

In [41]:
preference_point_columns = ["attr1_1", "sinc1_1", "intel1_1", "fun1_1", "amb1_1", "shar1_1"]
preference_point_sum_per_participant = (
    speed_dating_data.groupby("iid")[preference_point_columns].first().sum(axis=1, min_count=6)
)
scorecard_rating_columns = ["attr", "sinc", "intel", "fun", "amb", "shar", "like", "prob"]

print("missing pid", int(speed_dating_data["pid"].isna().sum()))
print("missing id", int(speed_dating_data["id"].isna().sum()))
print(
    "preference sum not 100",
    int((preference_point_sum_per_participant.dropna().sub(100).abs() > 1).sum()),
)
print("ratings equal to 0", int((speed_dating_data[scorecard_rating_columns] == 0).sum().sum()))
print("met equal to 0", int((speed_dating_data["met"] == 0).sum()))
print("met above 2", int((speed_dating_data["met"] > 2).sum()))


missing pid 10
missing id 1
preference sum not 100 9
ratings equal to 0 157
met equal to 0 4047
met above 2 8


9. Same text written in more than one way


In [43]:
import re
from collections import defaultdict

text_feature_columns = ["field", "career", "from", "undergra"]

def normalized_text(value):
    value = str(value).strip().lower().replace("&", " and ")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

for text_column in text_feature_columns:
    one_answer_per_participant = speed_dating_data.groupby("iid")[text_column].first().dropna()
    spellings_of_one_meaning = defaultdict(list)
    for raw_text, participant_count in one_answer_per_participant.value_counts().items():
        spellings_of_one_meaning[normalized_text(raw_text)].append((raw_text, int(participant_count)))
    different_spellings = {
        meaning: spellings
        for meaning, spellings in spellings_of_one_meaning.items()
        if len(spellings) > 1
    }
    print(text_column, "meanings written in more than one way:", len(different_spellings))
    for spellings in different_spellings.values():
        print(" ", [raw_text for raw_text, participant_count in spellings])

numeric_text_columns = ["mn_sat", "tuition", "income", "zipcode"]
for numeric_text_column in numeric_text_columns:
    text_values = speed_dating_data[numeric_text_column].dropna().astype(str)
    print(numeric_text_column, "values containing a comma:", int(text_values.str.contains(",").sum()))


field meanings written in more than one way: 37
  ['Business', 'business']
  ['Law', 'law', 'LAW']
  ['Social Work', 'social work', 'Social work']
  ['International Affairs', 'International affairs']
  ['Electrical Engineering', 'electrical engineering', 'ELECTRICAL ENGINEERING']
  ['Psychology', 'psychology']
  ['Engineering', 'engineering']
  ['Biology', 'biology']
  ['Film', 'film']
  ['Sociology', 'sociology']
  ['Business (MBA)', 'Business- MBA']
  ['Biomedical Engineering', 'biomedical engineering', 'Biomedical engineering']
  ['Political Science', 'political science']
  ['chemistry', 'Chemistry']
  ['Physics', 'physics']
  ['Epidemiology', 'epidemiology']
  ['Education', 'education', 'EDUCATION']
  ['medicine', 'Medicine']
  ['microbiology', 'Microbiology']
  ['Art Education', 'art education']
  ['Philosophy', 'philosophy']
  ['biotechnology', 'Biotechnology']
  ['Genetics', 'genetics']
  ['Elementary Education', 'elementary education']
  ['music education', 'Music Education']
 

10. Target distribution


In [42]:
print(speed_dating_data[target_column].value_counts().sort_index())
print(speed_dating_data[target_column].value_counts(normalize=True).sort_index())

dec
0    4860
1    3518
Name: count, dtype: int64
dec
0    0.580091
1    0.419909
Name: proportion, dtype: float64
